In [ ]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import joblib
import os

# Load the dataset
file_path = 'C:/Users/nosao/Desktop/Maxwell-Text Classification/Target Response/data/target_teamwork_motivation.csv' #Replace with actual file path
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()  # Clean column names

# Clean missing values
df = df.dropna(subset=['Job description', 'Question 7', 'Question 8', 'Question 9', 'Question 10', 'Question 11'])

# Convert target variables to strings
df['Question 7'] = df['Question 7'].astype(str)
df['Question 8'] = df['Question 8'].astype(str)
df['Question 9'] = df['Question 9'].astype(str)
df['Question 10'] = df['Question 10'].astype(str)
df['Question 11'] = df['Question 11'].astype(str)

# Extract job descriptions and target variables
X = df['Job description'].astype(str)
y_q7 = df['Question 7']
y_q8 = df['Question 8']
y_q9 = df['Question 9']
y_q10 = df['Question 10']
y_q11 = df['Question 11']

# Split data into training and test sets for each question
X_train, X_test, y_train_7, y_test_7 = train_test_split(X, y_q7, test_size=0.2, random_state=42)
X_train_q8, X_test_q8, y_train_q8, y_test_q8 = train_test_split(X, y_q8, test_size=0.2, random_state=42)
X_train_q9, X_test_q9, y_train_q9, y_test_q9 = train_test_split(X, y_q9, test_size=0.2, random_state=42)
X_train_q10, X_test_q10, y_train_q10, y_test_q10 = train_test_split(X, y_q10, test_size=0.2, random_state=42)
X_train_q11, X_test_q11, y_train_q11, y_test_q11 = train_test_split(X, y_q11, test_size=0.2, random_state=42)

# Define Logistic Regression with class_weight='balanced' to handle class imbalance
logreg_pipeline = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000, class_weight='balanced'))

# Hyperparameter tuning grid
param_grid_logreg = {
    'logisticregression__C': [0.01, 0.1, 1, 10],
    'tfidfvectorizer__ngram_range': [(1, 1), (1, 2), (1, 3)],
    'tfidfvectorizer__max_df': [0.85, 0.9, 0.95],
    'tfidfvectorizer__min_df': [1, 5],
    'tfidfvectorizer__use_idf': [True, False],
    'tfidfvectorizer__sublinear_tf': [True, False]
}

# Hyperparameter tuning with StratifiedKFold to ensure balanced class representation in cross-validation
cv = StratifiedKFold(n_splits=5)

# Hyperparameter tuning for Logistic Regression for Question 7
grid_logreg_q7 = GridSearchCV(logreg_pipeline, param_grid_logreg, cv=cv, scoring='accuracy', n_jobs=-1)
grid_logreg_q7.fit(X_train, y_train_7)

# Hyperparameter tuning for Logistic Regression for Question 8
grid_logreg_q8 = GridSearchCV(logreg_pipeline, param_grid_logreg, cv=cv, scoring='accuracy', n_jobs=-1)
grid_logreg_q8.fit(X_train_q8, y_train_q8)

# Hyperparameter tuning for Logistic Regression for Question 9
grid_logreg_q9 = GridSearchCV(logreg_pipeline, param_grid_logreg, cv=cv, scoring='accuracy', n_jobs=-1)
grid_logreg_q9.fit(X_train_q9, y_train_q9)

# Hyperparameter tuning for Logistic Regression for Question 10
grid_logreg_q10 = GridSearchCV(logreg_pipeline, param_grid_logreg, cv=cv, scoring='accuracy', n_jobs=-1)
grid_logreg_q10.fit(X_train_q10, y_train_q10)

# Hyperparameter tuning for Logistic Regression for Question 11
grid_logreg_q11 = GridSearchCV(logreg_pipeline, param_grid_logreg, cv=cv, scoring='accuracy', n_jobs=-1)
grid_logreg_q11.fit(X_train_q11, y_train_q11)

# Use the best Logistic Regression models for each question
best_logreg_model_q7 = grid_logreg_q7.best_estimator_
best_logreg_model_q8 = grid_logreg_q8.best_estimator_
best_logreg_model_q9 = grid_logreg_q9.best_estimator_
best_logreg_model_q10 = grid_logreg_q10.best_estimator_
best_logreg_model_q11 = grid_logreg_q11.best_estimator_

# Function to display metrics
def display_metrics(y_true, y_pred, question_num):
    print(f"Metrics for Question {question_num}")
    print(classification_report(y_true, y_pred))
    print(f"Accuracy: {accuracy_score(y_true, y_pred)}\n")

# Predict on the full test set for each question and display metrics
y_pred_q7 = best_logreg_model_q7.predict(X_test)
display_metrics(y_test_7, y_pred_q7, 7)

y_pred_q8 = best_logreg_model_q8.predict(X_test_q8)
display_metrics(y_test_q8, y_pred_q8, 8)

y_pred_q9 = best_logreg_model_q9.predict(X_test_q9)
display_metrics(y_test_q9, y_pred_q9, 9)

y_pred_q10 = best_logreg_model_q10.predict(X_test_q10)
display_metrics(y_test_q10, y_pred_q10, 10)

y_pred_q11 = best_logreg_model_q11.predict(X_test_q11)
display_metrics(y_test_q11, y_pred_q11, 11)


# Function to make predictions on a new job description
def predict_for_new_job_description(job_description):
    # Ensure the input is a string
    job_description = [job_description]

    # Predict for each question using the best model
    pred_q7 = best_logreg_model_q7.predict(job_description)[0]
    pred_q8 = best_logreg_model_q8.predict(job_description)[0]
    pred_q9 = best_logreg_model_q9.predict(job_description)[0]
    pred_q10 = best_logreg_model_q10.predict(job_description)[0]
    pred_q11 = best_logreg_model_q11.predict(job_description)[0]

    # Display or return the results
    print("Predictions for the new job description:")
    print(f"Question 7: {pred_q7}")
    print(f"Question 8: {pred_q8}")
    print(f"Question 9: {pred_q9}")
    print(f"Question 10: {pred_q10}")
    print(f"Question 11: {pred_q11}")

# Example: Enter a new job description
new_job_description = """
Purpose of the Role: To provide an effective Joinery resource to ensure the University
fabric is efficiently maintained on a day-to-day basis including undertaking Project works. To
ensure the effective interaction of Estate and Facilities services with other services.

Responsible to: Estates Team Leader

Main Duties and Responsibilities:
1. To provide all forms of Joinery duties and tasks in which you are competent within the
University Estate possessing at least five years of trade experience. Working with the team
across various other construction trades.
2. To be responsible for day-to-day breakdown and reactive maintenance.
3. To participate in the Maintenance call-out rota team.
4. To be responsible for working to and delivering cyclical maintenance works ensuring
certain activities are carried out as per the PPM regime.
5. To manage the fire door programme focussing on the Inspection, maintenance (ART
accepted repair techniques), and repair to achieve a compliant campus. This will involve
a good theoretical knowledge of the standards and regulations.
6. To be responsible for the Fire door dashboard, upkeep, and maintenance of the system.
To provide information on defects and repair (analyzing reports), accepted repair
techniques, and input for Projects.
7. To maintain Fire door records on CAFM and in line with the Fire Safety Regulations
(2023). Records must be kept.
8. Identify hazards, defects, and the need for adjustment or repair; to ensure compliance
with agreed codes, law, working practices, and health and safety whilst carrying out your
duties.
9. To provide support and guidance to Contractors engaged in Fire door campus works and
act as a focal point ensuring a fully compliant Fire door install is delivered to the Estate.
10. To manage quantities required to complete each task and manage material stocks and
ordering process.
11. To be responsible for ensuring all tools and equipment are maintained in good working
order and ready for use including power tools within the Estate.
12. To ensure all University fixtures, fittings, furniture, doors, locks, flooring, and other
Joinery items are efficiently maintained, repaired, constructed, or replaced, working
closely with Estates, Facilities Managers, and Estates Team Leader to achieve.
13. To ensure works are delivered in compliance with documented risk assessments and Method
statements, and responsible for the production and review of role-specific risk
assessments.
14. To be responsible for a high standard of conduct always working in a safe and
professional manner reporting any health and safety-related issues to the Estates Team
Leader immediately.
15. To be responsible for working to and delivering all works and repairs in a manner that
ensures VFM and quality finishes are implemented and maintained.
16. To assist the team and organization with general duties over and above your core skills.
Promote, develop and expand the business of our organization generally meeting set
targets.
17. To aid and advise the other members of the University Estates & Facilities staff including
porters and grounds staff as required.
18. To adhere to all organization policies and procedures.
19. To be responsible for continued professional development ensuring the post holder is
conversant and aware of current regulations, legislation, and approved industry
standards to their job role. You will have a basic awareness of Asbestos, CDM,
health and safety regulations.
20. To undertake small projects as reasonably required of the job role and advise all Estates
teams to deliver solutions and cost reductions to all joinery works on campus.

General Duties:
21. To ensure the use of data complies with current regulations, particularly those relating to
GDPR.
22. To comply with all health, safety, and wellbeing policies and procedures at all times and to
take responsibility for promoting and safeguarding the welfare and protection of others.
23. To advocate, promote, and advance equity and social justice within your work.
24. To carry out other duties, commensurate with the grade of the post, as may reasonably be
directed by your line manager after due consultation.
"""

# Function to enforce rule-based evaluation after predictions
def apply_rule_based_correction(predictions):
    """
    Apply rule-based correction to predictions.
    Ensures at least one question gets 'A' and updates preceding questions to 'D'.
    """
    # Question order
    questions = [7, 8, 9, 10, 11]
    
    # If 'A' is already in predictions, apply the rule
    if 'A' in predictions.values():
        a_index = list(predictions.values()).index('A')  # Find the first occurrence of 'A'
        
        # Set all preceding questions to 'D'
        for i in range(a_index):
            predictions[questions[i]] = 'D'
    
    else:
        # If no 'A' is assigned, select the best candidate
        for q in questions:
            if predictions[q] in ['B', 'C']:  # Prefer B over C
                predictions[q] = 'A'
                a_index = questions.index(q)
                
                # Set all previous questions to 'D'
                for i in range(a_index):
                    predictions[questions[i]] = 'D'
                break  # Ensure only one 'A' is assigned

    return predictions

# Function to make predictions on a new job description with rule enforcement
def predict_for_new_job_description(job_description):
    job_description = [job_description]  # Ensure it's a list for model input
    
    # Model predictions
    predictions = {
        7: best_logreg_model_q7.predict(job_description)[0],
        8: best_logreg_model_q8.predict(job_description)[0],
        9: best_logreg_model_q9.predict(job_description)[0],
        10: best_logreg_model_q10.predict(job_description)[0],
        11: best_logreg_model_q11.predict(job_description)[0]
    }
    
    # Apply rule-based correction
    corrected_predictions = apply_rule_based_correction(predictions)

    # Display results
    print("Corrected Predictions for the new job description:")
    for q, pred in corrected_predictions.items():
        print(f"Question {q}: {pred}")


# Call the function with the job description
predict_for_new_job_description(new_job_description)


# Create a folder for saving models
save_dir = "saved_models"
os.makedirs(save_dir, exist_ok=True)  # Create the folder 

# Save the models in the "saved_models" directory
joblib.dump(best_logreg_model_q7, os.path.join(save_dir, "model_q7.pkl"))
joblib.dump(best_logreg_model_q8, os.path.join(save_dir, "model_q8.pkl"))
joblib.dump(best_logreg_model_q9, os.path.join(save_dir, "model_q9.pkl"))
joblib.dump(best_logreg_model_q10, os.path.join(save_dir, "model_q10.pkl"))
joblib.dump(best_logreg_model_q11, os.path.join(save_dir, "model_q11.pkl"))

print("Models saved in the 'saved_models' folder.")



c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Metrics for Question 7
              precision    recall  f1-score   support

           A       0.50      0.50      0.50         2
           B       0.40      0.25      0.31         8
           D       0.42      0.56      0.48         9

    accuracy                           0.42        19
   macro avg       0.44      0.44      0.43        19
weighted avg       0.42      0.42      0.41        19

Accuracy: 0.42105263157894735

Metrics for Question 8
              precision    recall  f1-score   support

           B       0.89      0.80      0.84        10
           D       0.80      0.89      0.84         9

    accuracy                           0.84        19
   macro avg       0.84      0.84      0.84        19
weighted avg       0.85      0.84      0.84        19

Accuracy: 0.8421052631578947

Metrics for Question 9
              precision    recall  f1-score   support

           A       0.73      0.89      0.80         9
           C       0.00      0.00      0.00         2

c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i

In [1]:
import joblib
import os

# Define the saved models directory
save_dir = "saved_models"

# Load models correctly
best_logreg_model_q7 = joblib.load(os.path.join(save_dir, "model_q7.pkl"))
best_logreg_model_q8 = joblib.load(os.path.join(save_dir, "model_q8.pkl"))
best_logreg_model_q9 = joblib.load(os.path.join(save_dir, "model_q9.pkl"))
best_logreg_model_q10 = joblib.load(os.path.join(save_dir, "model_q10.pkl"))
best_logreg_model_q11 = joblib.load(os.path.join(save_dir, "model_q11.pkl"))

print("Models loaded successfully!")

Models loaded successfully!


In [7]:
# Example: Enter a new job description
new_job_description = """
JOB TITLE: DEPARTMENT: GRADE: HOURS OF WORK: RESPONSIBLE TO: RESPONSIBLE FOR: PURPOSE OF POST: . Web Services Manager Marketing, Communications and Recruitment 8a 35 hours per week, normally 9.00 am to 5.00 pm Monday to Friday with one hour for lunch. Holders of posts at this level are expected to work whatever hours are necessary for the full performance of the duties set out below, subject to a minimum of 35 hours per week. Head of Marketing and Recruitment Web Developer To manage the technical infrastructure / framework for the delivery of Web Services and to be responsible for the development and support of the externally facing website, manage the Web Content Management System (WCMS) and web related tools and technologies used to deliver the Leeds Trinity Website and its user experience in line with user requirements and institutional objectives. MAIN DUTIES AND RESPONSIBLITIES: 1. Lead and manage the institutional Web Content Management System (CMS) on a day to day basis including the development of responsive style and layout templates, the integration of the CMS with other corporate systems (APIs), and the management of system users. 2. Lead, manage, develop and deploy bespoke code e.g. HTML/HTML5, CSS/SASS, PHP, JQuery, JSON, JavaScript, for the creation and management of reliable, high-performing, SEO optimised, user-oriented, secure, accessible, website designs, functionality and features. 3. Lead on the development and communication of the University's digital governance framework including the development and maintenance of a University Pattern Library. 4. Be responsible for developing and utilising a portfolio of web tools to meet the University’s business requirements and bring the highest degree of benefit to the website users experience. 5. Be the main point of contact and advice for all University staff for the tools and technologies used to deliver the externally facing website. 6. Advise on and plan for future web services developments, including preparation of requirements specifications. 7. Lead on the provision of reports and dashboards for web users and the Leadership and Executive team. 8. Be responsible for the development and implementation of the University’s Web Strategy and plans. 9. Oversee and coordinate the gathering of user requirements and subsequent user testing for new and existing websites and applications. 10. Lead on the use and development of appropriate design patterns and frameworks within web and digital projects. 11. Use analytics to understand user activity and improve the usability and functionality of the website. 12. Manage the web budget including forecasting, purchasing and monitoring. 13. Document code to aid future developments, additional contributions, maintenance and knowledge transfer across the University. 14. Document systems and processes pertaining to the Website, the CMS and other related web tools and systems as part of the Digital Service Manual. 15. Work collaboratively with the other members of the Marketing, Recruitment and Communications Team to strategically plan the website roadmap and continuously develop and improve working processes within the team. 16. Liaise with external suppliers (such as the CMS governance vendor) to plan and manage system upgrades, support requests and product enhancement suggestions. 17. Responsible for the development and maintenance of associated systems and technologies, including the University’s Search Tool and Asset Management Library. 18. Coordinate web support duties with other Digital Team members, supporting and assisting other team members with their duties as required. 19. Ongoing development of own web and digital skills, knowledge and web-related tools. to enable the University to be at the forefront of utilising new technologies, trends and methods. 20. Work with IT services and other corporate system colleagues to ensure: agreement of and adherence to development and documentation standards, effective systems change control and release management, effective management of the boundaries and interfaces between systems, secure and resilient system hosting, compliance with Information and IT Security policies and consistency of design patterns across University systems. 21. Contribution of skills and expertise to University groups and projects as required. 22. Provide CMS training to Marketing and Campaigns staff. 23. Represent the University on web services groups regionally and nationally. Leadership and Staff Management 24. Provide leadership and management for the Web Developer, providing training and development of their skills. 25. Provide innovative digital leadership to the University to promote and drive transformational change. 26. To promote team-work and inter-professional working. General Duties 27. To ensure the use of data complies with current regulations, particularly those relating to GDPR. 28. To comply with current health and safety requirements, work with relevant University policies and participate fully in the annual staff review scheme. 29. To apply the University’s Equality, Diversity and Inclusion Policy in the postholder’s area of responsibility and in their general conduct. 30. To carry out any other duties commensurate with the grade of the post as may reasonably be directed by the Deputy Director of Marketing, Communications and Recruitment after due consultation. This job description is current on the date indicated below. It is liable to variation by the Vice Chancellor in order to reflect or anticipate institutional developments and changes in the post. July 2021
"""

# Function to enforce rule-based evaluation after predictions
def apply_rule_based_correction(predictions):
    """
    Apply rule-based correction to predictions.
    Ensures at least one question gets 'A' and updates preceding questions to 'D'.
    """
    # Question order
    questions = [7, 8, 9, 10, 11]
    
    # If 'A' is already in predictions, apply the rule
    if 'A' in predictions.values():
        a_index = list(predictions.values()).index('A')  # Find the first occurrence of 'A'
        
        # Set all preceding questions to 'D'
        for i in range(a_index):
            predictions[questions[i]] = 'D'
    
    else:
        # If no 'A' is assigned, select the best candidate
        for q in questions:
            if predictions[q] in ['B', 'C']:  # Prefer B over C
                predictions[q] = 'A'
                a_index = questions.index(q)
                
                # Set all previous questions to 'D'
                for i in range(a_index):
                    predictions[questions[i]] = 'D'
                break  # Ensure only one 'A' is assigned

    return predictions

# Function to make predictions on a new job description with rule enforcement
def predict_for_new_job_description(job_description):
    job_description = [job_description]  # Ensure it's a list for model input
    
    # Model predictions
    predictions = {
        7: best_logreg_model_q7.predict(job_description)[0],
        8: best_logreg_model_q8.predict(job_description)[0],
        9: best_logreg_model_q9.predict(job_description)[0],
        10: best_logreg_model_q10.predict(job_description)[0],
        11: best_logreg_model_q11.predict(job_description)[0]
    }
    
    # Apply rule-based correction
    corrected_predictions = apply_rule_based_correction(predictions)

    # Display results
    print("Corrected Predictions for the new job description:")
    for q, pred in corrected_predictions.items():
        print(f"Question {q}: {pred}")


# Call the function with the job description
predict_for_new_job_description(new_job_description)


Corrected Predictions for the new job description:
Question 7: A
Question 8: B
Question 9: A
Question 10: D
Question 11: D
